# News Headline Generation
RNN, LSTM, Transformer from scratch...

In [ ]:
import sys
sys.path.append("src")

from dtypes.config import DatasetConfig, TokenConfig, ModelConfig, TrainConfig
from dtypes.enums import ModelName
from wrappers.make import DataBuild, TokBuild, ModelBuild, TrainBuild

In [ ]:
dcfg = DatasetConfig("dataset/news.csv")
tcfg = TokenConfig(1)
mcfg = ModelConfig(ModelName.LSTM, 128, 256, 1, 4, 256)
trcfg = TrainConfig(8, 0.001, 1, 1.0, 0.5, 7)

In [ ]:
data = DataBuild().build(dcfg)
tok = TokBuild().build(tcfg)
tok.build_vocab(data.vocab_texts())
model = ModelBuild().build(mcfg, tok.size())
tr = TrainBuild().build(trcfg)
tr.fit(model, data, tok)

In [ ]:
from wrappers import engine as en

dev = en.device()
model.to(dev)
data.build(tok)
tr_idx, te_idx = data.split()
if len(te_idx) == 0:
    te_idx = tr_idx
b = [te_idx[0]]
batch = data.get_batch(b)
art = en.tensor(batch.art_ids, dtype=en.long, device=dev)
pred = model.generate(art, data.cfg.max_sum_len)
pid = pred[0].tolist()
out = tok.decode(pid)
out